In [20]:
import nltk
import random
from nltk.corpus import movie_reviews
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score
import pandas as pd
import string

nltk.download('movie_reviews')
nltk.download('stopwords')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
rows = [
    {
        "text":  " ".join(movie_reviews.words(fileid)),
        "label": label
    }
    for label in movie_reviews.categories()          
    for fileid in movie_reviews.fileids(label)
]

df = pd.DataFrame(rows)
df.head()

,text,label
0,"plot : two teen couples go to a church party ,...",neg
1,the happy bastard ' s quick movie review damn ...,neg
2,it is movies like these that make a jaded movi...,neg
3,""" quest for camelot "" is warner bros . ' first...",neg
4,synopsis : a mentally unstable man undergoing ...,neg


In [13]:
def text_preprocesssor(text):
    # remove punctuation
    text = "".join([c for c in text if c not in string.punctuation])

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords
    stop_words = stopwords.words("english")
    stop_words.append("url")  # add custom stopword
    tokens = [w for w in tokens if w not in stop_words]

    # lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    # return final cleaned text
    return " ".join(tokens)

In [15]:
df["cleaned_text"] = df["text"].apply(text_preprocesssor)
df.head()

,text,label,cleaned_text
0,"plot : two teen couples go to a church party ,...",neg,plot two teen couple go church party drink dri...
1,the happy bastard ' s quick movie review damn ...,neg,happy bastard quick movie review damn y2k bug ...
2,it is movies like these that make a jaded movi...,neg,movie like make jaded movie viewer thankful in...
3,""" quest for camelot "" is warner bros . ' first...",neg,quest camelot warner bros first feature length...
4,synopsis : a mentally unstable man undergoing ...,neg,synopsis mentally unstable man undergoing psyc...


In [24]:
X = df["cleaned_text"]
vectorizer = TfidfVectorizer()
x_vectorized = vectorizer.fit_transform(X)

df["label"] = df["label"].replace({"neg":0, "pos":1})
y= df["label"]

# Train test split
X_train, X_test, y_train, y_test = train_test_split(x_vectorized, 
                                                    y, test_size=0.2, 
                                                    random_state=42
                                                    )

In [25]:
# Model training
model = MultinomialNB()
model.fit(X_train, y_train)

# Model evaluation
y_pred = model.predict(X_test)
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision}")

Precision: 0.85


In [ ]:
# Test
review = "The movie did not amaze me."
review_processed = text_preprocesssor(review)
review_vectorized = vectorizer.transform([review_processed])
prediction = model.predict(review_vectorized)
print("Prediction: ", )

Prediction:  [0]


In [27]:
def movie_review_sentiment(review):
    review_processed = text_preprocesssor(review)
    review_vectorized = vectorizer.transform([review_processed])
    prediction = model.predict(review_vectorized)
    return "Positive" if prediction[0] == 1 else "Negative"

In [28]:
movie_review_sentiment("The movie was fantastic and I loved it!")

'Positive'